In [1]:
# pip install pypdf faiss-cpu sentence-transformers numpy

In [2]:
from pypdf import PdfReader

def load_pdf_text(path: str) -> str:
    reader = PdfReader(path)
    pages = []
    for i, p in enumerate(reader.pages):
        t = p.extract_text() or ""
        pages.append(t)
    return "\n".join(pages)

def chunk_text(text: str, chunk_size=1200, overlap=200):
    # chunk_size/overlap are characters here (simple MVP)
    chunks = []
    start = 0
    while start < len(text):
        end = min(len(text), start + chunk_size)
        chunks.append(text[start:end].strip())
        start = max(end - overlap, 0)
        if end == len(text):
            break
    return [c for c in chunks if c]

In [3]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def embed_texts(texts):
    embs = embed_model.encode(texts, normalize_embeddings=True)
    return np.array(embs, dtype=np.float32)

pdf_path = "My-international-recipes-Nobilia-K-chen.pdf"  # change this
raw = load_pdf_text(pdf_path)
chunks = chunk_text(raw)

embs = embed_texts(chunks)
index = faiss.IndexFlatIP(embs.shape[1])   # cosine similarity with normalized vectors
index.add(embs)

print("Chunks:", len(chunks), "Embedding dim:", embs.shape[1])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\kinst\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\kinst\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Chunks: 178 Embedding dim: 384


In [4]:
def retrieve(query: str, top_k=4):
    q_emb = embed_texts([query])
    scores, idxs = index.search(q_emb, top_k)
    results = []
    for score, idx in zip(scores[0], idxs[0]):
        if idx == -1: 
            continue
        results.append({"score": float(score), "chunk": chunks[idx]})
    return results

In [ ]:
import requests, json, re

def qwen_generate_ollama(prompt: str, model="qwen2.5:7b-instruct"):
    # You must have: ollama serve running, and model pulled
    url = "http://localhost:11434/api/generate"
    payload = {"model": model, "prompt": prompt, "stream": False}
    r = requests.post(url, json=payload, timeout=120)
    r.raise_for_status()
    return r.json()["response"]

In [12]:
def retrieve_two_stage(user_ingredients, user_prefs="", top_k1=4, top_k2=4, mode="ollama"):
    # Stage 1: retrieve by ingredients
    q1 = " ".join(user_ingredients + ([user_prefs] if user_prefs else []))
    stage1 = retrieve(q1, top_k=top_k1)

    # Extract techniques from stage1
    techniques = extract_techniques_from_chunks(user_ingredients, stage1, mode=mode)

    # Stage 2: retrieve by technique tags (if any)
    if techniques:
        q2 = " ".join(techniques + ([user_prefs] if user_prefs else []))
        stage2 = retrieve(q2, top_k=top_k2)
    else:
        stage2 = []

    # Merge (dedupe by chunk text)
    seen = set()
    merged = []
    for r in stage1 + stage2:
        key = r["chunk"][:200]
        if key in seen:
            continue
        seen.add(key)
        merged.append(r)

    return merged, techniques

In [ ]:
def parse_json_safely(text: str):
    text = (text or "").strip()
    if not text:
        return None
    try:
        return json.loads(text)
    except:
        m = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if m:
            try: return json.loads(m.group(0))
            except: return None
    return None

def extract_techniques_from_chunks(user_ingredients, retrieved_chunks, mode="ollama"):
    context_preview = "\n\n---\n\n".join([r["chunk"][:500] for r in retrieved_chunks])

    prompt = f"""
Return ONLY JSON.

User ingredients: {user_ingredients}

From the context below, propose 3-6 cooking technique tags that would help retrieval.
Tags should be short phrases like:
"stir-fry", "fried rice", "marinade", "one-pan", "pan-sear", "egg scramble", "rice bowl"

CONTEXT:
{context_preview}

Output JSON schema:
{{"techniques":["tag1","tag2","tag3"]}}
"""
    if mode == "ollama":
        raw = qwen_generate_ollama(prompt)
    else:
        raw = qwen_generate_llamacpp(prompt)

    data = parse_json_safely(raw)
    return data.get("techniques", []) if data else []

In [14]:
def build_hybrid_recipe_prompt(user_ingredients, user_prefs, retrieved_chunks):
    context = "\n\n---\n\n".join([r["chunk"] for r in retrieved_chunks])

    return f"""
You are a practical cooking assistant.

Goal:
- Create ONE recipe that the user can actually cook using their available ingredients.
- Use the PDF context as a trusted reference for cooking techniques and realistic steps.

Hard Rules:
1) You MUST prioritize using the user's ingredients (Required).
2) You MAY use only these common staples (assume user has): oil, salt, pepper, soy sauce, garlic.
3) You MAY suggest extra ingredients ONLY as "Optional add-ons" (clearly marked).
4) Do NOT make any optional add-on required.
5) If the PDF context does not contain a close recipe match, still produce a realistic recipe using general techniques found in the context (e.g., stir-fry, fried rice, pan-sear).

User ingredients (must use as much as possible):
{user_ingredients}

User preferences:
{user_prefs}

PDF context (trusted reference):
{context}

Output format (MUST follow exactly):

Title:
Why this recipe fits my ingredients:
Ingredients (Required):
Ingredients (Staples assumed):
Ingredients (Optional add-ons):
Steps:
Time:
Substitutions (if any):
Notes:
""".strip()

In [10]:
def rag_recipe_hybrid(user_ingredients, user_prefs="", top_k1=4, top_k2=4, mode="ollama"):
    retrieved, techniques = retrieve_two_stage(
        user_ingredients, user_prefs=user_prefs,
        top_k1=top_k1, top_k2=top_k2, mode=mode
    )

    print("\n=== TECHNIQUE TAGS (auto) ===")
    print(techniques)

    print("\n=== RETRIEVED CHUNKS (preview) ===")
    for i, r in enumerate(retrieved, 1):
        print(f"\n[{i}] score={r['score']:.3f}\n{r['chunk'][:350]}...")

    prompt = build_hybrid_recipe_prompt(user_ingredients, user_prefs, retrieved)

    print("\n=== GENERATING (Hybrid RAG) ===")
    if mode == "ollama":
        return qwen_generate_ollama(prompt)
    else:
        return qwen_generate_llamacpp(prompt)

In [15]:
ans = rag_recipe_hybrid(
    user_ingredients=["chicken breast", "rice", "egg"],
    user_prefs="quick 15 minutes, Malaysian style",
    mode="ollama"
)
print("\n=== FINAL ANSWER ===\n", ans)


=== TECHNIQUE TAGS (auto) ===
['stir-fry', 'scrambled eggs', 'one-pan']

=== RETRIEVED CHUNKS (preview) ===

[1] score=0.475
or until softened but not 
coloured. 
2. Add the rice to the pan and stir around until it is mixed with the vegetables and heated 
through, then push to the side of the wok. Pour the eggs into the centre of the pan and stir 
until just set. 
3. Once the eggs are scrambled, toss all the ingredients together, add the soy sauce, and 
serve immediately...

[2] score=0.469
es, peppercorns, and a pinch of salt. Pour in enough water to just cover the 
chicken. Bring everything to the boil and simmer for about 1½ hours until the meat is tender 
and comes away from the bones. 
2. Lift the chicken out of the pot and let it cool. Take the meat off the bones and tear it into 
pieces. Pour the stock through a sieve set over ...

[3] score=0.457
es, finely 
chopped 
300g (10½oz)   Arborio rice 
  650ml (1–1¼pt) hot 
chicken stock 
200g (7oz)   cooked chicken, torn 
into bite-